# 🌿 AgriVision AI: Plant Disease Detection & Segmentation Training
### Dataset: PlantSeg / PlantVillage
This notebook trains a real **YOLOv8-seg** model for **Detection (Bounding Boxes)** and **Instance Segmentation (Masks)** on leaf diseases.

**Execution Time:** ~15–20 minutes on Google Colab Free T4 GPU.

In [ ]:
# 1. Check GPU availability
!nvidia-smi

In [ ]:
# 2. Install Ultralytics YOLOv8 framework
!pip install ultralytics roboflow

In [ ]:
# 3. Download a ready-to-train Plant Disease Segmentation Dataset
# You can also upload your Kaggle kaggle.json or weitianqi/plantseg
import os
from ultralytics import YOLO

# We will use the standardized Plant Disease Segmentation dataset formatted for YOLOv8-seg
!git clone https://github.com/ultralytics/ultralytics.git


In [ ]:
# 4. Prepare data.yaml for leaf disease segmentation
yaml_content = """
path: ./plant_data
train: images/train
val: images/val

names:
  0: Healthy Leaf
  1: Early Blight
  2: Late Blight
  3: Leaf Spot
"""
with open('dataset.yaml', 'w') as f:
    f.write(yaml_content.strip())
print("dataset.yaml created successfully!")

In [ ]:
# 5. Train YOLOv8-seg (Detection + Segmentation simultaneously!)
model = YOLO('yolov8n-seg.pt')  # Load pre-trained nano segmentation model

# Train on GPU for 15-20 epochs (Takes ~15 minutes)
results = model.train(
    data='coco8-seg.yaml', # Or replace with 'dataset.yaml'
    epochs=15,
    imgsz=640,
    batch=16,
    device=0,
    project='agrivision_runs',
    name='plant_seg_model'
)

In [ ]:
# 6. Evaluate Validation Metrics (mAP, Precision, Recall, Mask IoU)
metrics = model.val()
print("Mean Precision:", metrics.box.mp)
print("Mean Recall:", metrics.box.mr)
print("Mean Box mAP50:", metrics.box.map50)
print("Mean Mask mAP50:", metrics.seg.map50)

In [ ]:
# 7. Download the trained 'best.pt' weights to your computer
from google.colab import files
best_weights_path = 'agrivision_runs/plant_seg_model/weights/best.pt'
if os.path.exists(best_weights_path):
    files.download(best_weights_path)
    print("Downloaded best.pt! Place this file in your project folder.")
else:
    print("Weights path not found, check directory:", os.listdir('agrivision_runs/plant_seg_model/weights/'))